# Backpropagation, by Hand: The Chain Rule, Run Backward

The full code of [Backpropagation, by Hand: The Chain Rule, Run Backward](https://residual-stream.lvtan71.workers.dev/posts/backpropagation-by-hand/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/backpropagation-by-hand) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/backpropagation-by-hand.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 13 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 12 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "backpropagation-by-hand"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/backpropagation-by-hand


## `xor.py`

The post's hand calculation (one XOR example, every gradient, one step), then how often training finds XOR from 100 random starts at hidden widths 2 to 16.

In [2]:
!{sys.executable} xor.py

z1 [ 0.5 -0.3]  a1 [0.5 0. ]  z2 0.4500  p 0.6106  loss 0.4932
dL/dw2 = [-0.1947, 0.0]
dL/db2 = -0.3894
dL/dW1 = [[-0.2726, 0.0], [0.0, 0.0]]
dL/db1 = [-0.2726, 0.0]
after one step: p 0.7131  loss 0.3381


 2 hidden units: XOR solved from 18/100 random starts


 3 hidden units: XOR solved from 52/100 random starts


 4 hidden units: XOR solved from 73/100 random starts


 8 hidden units: XOR solved from 96/100 random starts


16 hidden units: XOR solved from 100/100 random starts


The post printed:

```text
z1 [ 0.5 -0.3]  a1 [0.5 0. ]  z2 0.4500  p 0.6106  loss 0.4932
dL/dw2 = [-0.1947, 0.0]
dL/db2 = -0.3894
dL/dW1 = [[-0.2726, 0.0], [0.0, 0.0]]
dL/db1 = [-0.2726, 0.0]
after one step: p 0.7131  loss 0.3381
 2 hidden units: XOR solved from 18/100 random starts
 3 hidden units: XOR solved from 52/100 random starts
 4 hidden units: XOR solved from 73/100 random starts
 8 hidden units: XOR solved from 96/100 random starts
16 hidden units: XOR solved from 100/100 random starts
```

## `xor_failures.py`

The 82 two-unit starts that fail, classified: right answers, final loss, whether a hidden unit was dead from step 0, and the gradient's size.

In [3]:
!{sys.executable} xor_failures.py

starts with a unit dead from step 0: 42/100, of which solved: 0
 27 failures: 3/4 right, loss 0.477, dead unit at start, gradient < 1e-3
 22 failures: 3/4 right, loss 0.347, both units alive, gradient < 1e-3
 15 failures: 2/4 right, loss 0.693, dead unit at start, gradient < 1e-3
 14 failures: 3/4 right, loss 0.477, both units alive, gradient < 1e-3
  3 failures: 2/4 right, loss 0.693, both units alive, gradient < 1e-3
  1 failures: 3/4 right, loss 0.347, both units alive, gradient >= 1e-3


The post printed:

```text
starts with a unit dead from step 0: 42/100, of which solved: 0
 27 failures: 3/4 right, loss 0.477, dead unit at start, gradient < 1e-3
 22 failures: 3/4 right, loss 0.347, both units alive, gradient < 1e-3
 15 failures: 2/4 right, loss 0.693, dead unit at start, gradient < 1e-3
 14 failures: 3/4 right, loss 0.477, both units alive, gradient < 1e-3
  2 failures: 2/4 right, loss 0.693, both units alive, gradient < 1e-3
  1 failures: 3/4 right, loss 0.347, both units alive, gradient >= 1e-3
  1 failures: 3/4 right, loss 0.693, both units alive, gradient < 1e-3
```

## `grad_check.py`

Backpropagation against central finite differences on a small random problem.

In [4]:
!{sys.executable} grad_check.py

largest relative error, backprop vs finite differences: 9.7e-08


The post printed:

```text
largest relative error, backprop vs finite differences: 9.7e-08
```

## `router_mlp.py`

The router with 16 hidden units and five L2 penalties against the linear router, on 20 splits, paired differences with the corrected standard error, then the test score of the network chosen on validation.

In [5]:
!{sys.executable} router_mlp.py

                      train  val loss  vs linear 0.003  val acc
linear, L2 0.003      0.065     0.347                     0.866
linear, L2 0.01       0.141     0.346   -0.001 ± 0.021    0.861
16 hidden, L2 0.0     0.000     0.689   +0.343 ± 0.143    0.872
16 hidden, L2 0.003   0.010     0.416   +0.069 ± 0.040    0.868
16 hidden, L2 0.01    0.032     0.365   +0.018 ± 0.015    0.869
16 hidden, L2 0.03    0.098     0.342   -0.005 ± 0.011    0.866
16 hidden, L2 0.1     0.314     0.399   +0.053 ± 0.047    0.834

chosen on validation: 16 hidden, L2 0.03. test loss 0.332 vs linear 0.333, difference -0.002 ± 0.008


The post printed:

```text
                      train  val loss  vs linear 0.003  val acc
linear, L2 0.003      0.065     0.347                     0.866
linear, L2 0.01       0.141     0.346   -0.001 ± 0.021    0.861
16 hidden, L2 0.0     0.000     0.689   +0.343 ± 0.143    0.872
16 hidden, L2 0.003   0.010     0.416   +0.069 ± 0.040    0.868
16 hidden, L2 0.01    0.032     0.365   +0.018 ± 0.015    0.869
16 hidden, L2 0.03    0.098     0.342   -0.005 ± 0.011    0.866
16 hidden, L2 0.1     0.314     0.399   +0.053 ± 0.047    0.834

chosen on validation: 16 hidden, L2 0.03. test loss 0.332 vs linear 0.333, difference -0.002 ± 0.008
```

## `steps_and_seeds.py`

The network at 3,000 and 12,000 steps and from five initialization seeds per split, with and without the penalty.

In [6]:
!{sys.executable} steps_and_seeds.py

L2 0.0   3,000 steps: val loss 0.677-0.692 across 5 initialization seeds
L2 0.0  12,000 steps: val loss 0.778-0.795 across 5 initialization seeds
L2 0.03  3,000 steps: val loss 0.342-0.343 across 5 initialization seeds
L2 0.03 12,000 steps: val loss 0.342-0.342 across 5 initialization seeds


The post printed:

```text
L2 0.0   3,000 steps: val loss 0.677-0.692 across 5 initialization seeds
L2 0.0  12,000 steps: val loss 0.778-0.795 across 5 initialization seeds
L2 0.03  3,000 steps: val loss 0.342-0.343 across 5 initialization seeds
L2 0.03 12,000 steps: val loss 0.342-0.342 across 5 initialization seeds
```